<a href="https://colab.research.google.com/github/kej534923-maker/ECON5200-Applied-Data-Analytics/blob/main/forecast_evaluation_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
forecast_evaluation.py — Forecast Evaluation & Backtesting Module

Reusable functions for:
- Mean Absolute Scaled Error (MASE)
- Expanding-window backtesting

Author: [Your Name]
Course: ECON 5200
"""

import numpy as np
import pandas as pd
from typing import Callable


def compute_mase(
    actual: np.ndarray,
    forecast: np.ndarray,
    insample: np.ndarray,
    m: int = 1
) -> float:
    """Compute Mean Absolute Scaled Error (MASE).

    MASE = MAE(forecast) / MAE(naive seasonal forecast on in-sample data)

    Args:
        actual: True values (out-of-sample)
        forecast: Predicted values
        insample: Training data
        m: Seasonal period (1=random walk, 12=monthly)

    Returns:
        float: MASE score
    """
    actual = np.asarray(actual).ravel()
    forecast = np.asarray(forecast).ravel()
    insample = np.asarray(insample).ravel()

    if len(actual) == 0 or len(forecast) == 0:
        raise ValueError("actual and forecast cannot be empty.")
    if len(actual) != len(forecast):
        raise ValueError("actual and forecast must have same length.")
    if len(insample) <= m:
        raise ValueError("insample must be longer than seasonal period m.")

    # Forecast error
    mae_forecast = np.mean(np.abs(actual - forecast))

    # Naive seasonal error
    naive_errors = insample[m:] - insample[:-m]
    mae_naive = np.mean(np.abs(naive_errors))

    if np.isclose(mae_naive, 0):
        raise ValueError("Naive MAE is zero — cannot scale.")

    return float(mae_forecast / mae_naive)


def backtest_expanding_window(
    series: pd.Series,
    model_fn: Callable,
    min_train: int = 120,
    horizon: int = 12,
    step: int = 12
) -> pd.DataFrame:
    """Expanding-window backtest for time series models.

    Args:
        series: Full time series (pd.Series with datetime index)
        model_fn: Function(train_series) -> forecast (length=horizon)
        min_train: Minimum training size
        horizon: Forecast horizon
        step: Step size between re-trains

    Returns:
        pd.DataFrame: backtest results
    """
    if not isinstance(series, pd.Series):
        raise TypeError("series must be a pandas Series.")

    series = series.dropna()

    if len(series) < min_train + horizon:
        raise ValueError("Series too short for backtesting.")

    results = []

    for origin in range(min_train, len(series) - horizon + 1, step):
        train = series.iloc[:origin]
        test = series.iloc[origin:origin + horizon]

        forecast = model_fn(train)
        forecast = np.asarray(forecast).ravel()

        if len(forecast) != horizon:
            raise ValueError("model_fn must return forecast of correct length.")

        actual = test.values
        error = actual - forecast
        abs_error = np.abs(error)

        mase_value = compute_mase(
            actual=actual,
            forecast=forecast,
            insample=train.values,
            m=1
        )

        for h in range(horizon):
            results.append({
                "origin": train.index[-1],
                "horizon": h + 1,
                "actual": actual[h],
                "forecast": forecast[h],
                "error": error[h],
                "abs_error": abs_error[h],
                "mase": mase_value
            })

    return pd.DataFrame(results)


if __name__ == "__main__":
    print("forecast_evaluation.py loaded successfully.")

    # Quick test
    actual = np.array([10, 12, 14])
    forecast = np.array([11, 11, 15])
    insample = np.arange(1, 20)

    print("Test MASE:", compute_mase(actual, forecast, insample))

    def dummy_model(train):
        return np.repeat(train.iloc[-1], 3)

    series = pd.Series(
        np.arange(1, 40),
        index=pd.date_range("2020-01-01", periods=39, freq="MS")
    )

    bt = backtest_expanding_window(series, dummy_model, 12, 3, 3)
    print(bt.head())

forecast_evaluation.py loaded successfully.
Test MASE: 1.0
      origin  horizon  actual  forecast  error  abs_error  mase
0 2020-12-01        1      13        12      1          1   2.0
1 2020-12-01        2      14        12      2          2   2.0
2 2020-12-01        3      15        12      3          3   2.0
3 2021-03-01        1      16        15      1          1   2.0
4 2021-03-01        2      17        15      2          2   2.0
